# TRABALHO PRÁTICO: CONSTRUÇÃO DE MODELO DE CLUSTERIZAÇÃO EM DADOS MASSIVOS COM SPARK ML

**Curso:** Ciência de Dados e Inteligência Artificial  
**Disciplina:** Processamento de Dados Massivos  
**Professor:** Prof. Alexandre Roriz  
**Aluno:** Eduardo Alves de Araújo  

---

## 1. CONTEXTUALIZAÇÃO E APRESENTAÇÃO DA BASE DE DADOS

### 1.1 Contextualização do Projeto
Este trabalho foi desenvolvido no âmbito da disciplina de Processamento de Dados Massivos com o objetivo de aplicar técnicas de engenharia de dados e aprendizado de máquina não supervisionado sobre uma base pública de grande escala, utilizando o ecossistema **Apache Spark** através da API **PySpark**.

Para a realização do projeto, foi selecionada a base pública **SCR.data**, disponibilizada pelo **Banco Central do Brasil (BCB)** através do seu Portal de Dados Abertos.

O **SCR.data** reúne informações agregadas das operações de crédito reportadas mensalmente pelas instituições financeiras ao Sistema de Informações de Créditos (SCR), permitindo acompanhar a evolução e as dinâmicas do mercado de crédito brasileiro. A base contempla indicadores cruciais como **carteira ativa**, **carteira vencida**, **inadimplência** e **ativos problemáticos**.

Além dos indicadores financeiros, a base oferece ricas dimensões categóricas para segmentação, incluindo unidade da federação (UF), perfil do cliente (Pessoa Física ou Jurídica), segmento financeiro, modalidade e submodalidade de crédito, porte econômico, origem dos recursos e indexadores das operações.

Para este projeto, são utilizados os arquivos oficiais da **Versão 2 do SCR.data** referentes ao biênio **2024 e 2025**.

---

### 1.2 Fonte dos Dados e Links para Reprodução

* **Portal de Dados Abertos:** https://dadosabertos.bcb.gov.br/dataset/scr_data
* **Arquivos Oficiais Utilizados (Versão 2):**
  * Dados Anuais 2024: https://www.bcb.gov.br/pda/desig/scrdata_2024.zip
  * Dados Anuais 2025: https://www.bcb.gov.br/pda/desig/scrdata_2025.zip

---

### 1.3 Pergunta de Negócio
> **"É possível identificar e caracterizar perfis distintos de comportamento e risco das operações de crédito brasileiras a partir dos indicadores de inadimplência, ativos problemáticos e volume de carteira divulgados pelo Banco Central do Brasil?"**

---

### 1.4 Justificativa do Cenário de Dados Massivos (Big Data)
A base **SCR.data** possui abrangência nacional e consolida dados de todo o Sistema Financeiro Nacional. O processo de ingestão e análise envolve a unificação de 24 arquivos mensais históricos correspondentes ao biênio 2024–2025, envolvendo um volume massivo de dados oriundos do Sistema Financeiro Nacional.

O volume de dados exige processamento distribuído porque:
1. **Memória e Custo Computacional:** A leitura, transformação e e execução de algoritmos de aprendizado de máquina sobre grandes volumes de dados impõem elevado custo computacional e de memória para abordagens tradicionais executadas em nó único e de execução sequencial de bibliotecas convencionais em nó único (como Pandas/Scikit-Learn).
2. **Escalabilidade:** O uso do PySpark viabiliza a execução de transformações paralelas no formato *DataFrame* do Spark, a aplicação de Pipelines padronizados e a execução distribuída da varredura de clusters ($K$).

*Obs: As características estruturais da base, incluindo quantidade de registros, número de colunas e estatísticas descritivas iniciais, serão apresentadas na etapa de ingestão e exploração dos dados.*

In [1]:
# ---------------------------------------------------------------------
# ETAPA 1: CONFIGURAÇÃO DO AMBIENTE SPARK
# ---------------------------------------------------------------------
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = SparkSession.builder \
    .appName("PySpark_KMeans_SCR_BCB_V2") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ SparkSession iniciada com sucesso! Versão PySpark: {spark.version}")

✅ SparkSession iniciada com sucesso! Versão PySpark: 4.0.5


In [2]:
# =====================================================================
# ETAPA 2: INGESTÃO E AUDITORIA INICIAL DA BASE SCR.DATA (VERSÃO 2)
# =====================================================================

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Configuração/Recuperação da SparkSession
spark = SparkSession.builder \
    .appName("PySpark_KMeans_SCR_BCB_V2") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# 2. Download direto e automatizado a partir da fonte oficial do BCB
print("⏳ Baixando repositórios oficiais scrdata_2024.zip e scrdata_2025.zip...")
!wget -q "https://www.bcb.gov.br/pda/desig/scrdata_2024.zip" -O /content/scrdata_2024.zip
!wget -q "https://www.bcb.gov.br/pda/desig/scrdata_2025.zip" -O /content/scrdata_2025.zip

# Descompactação em diretório dedicado
!unzip -q -o /content/scrdata_2024.zip -d /content/scr_v2/
!unzip -q -o /content/scrdata_2025.zip -d /content/scr_v2/

# Identificação dos arquivos CSV descompactados
pasta_v2 = "/content/scr_v2/"
arquivos_csv = [os.path.join(pasta_v2, f) for f in os.listdir(pasta_v2) if f.endswith('.csv') or f.endswith('.tsv') or f.endswith('.txt')]

print(f"✅ Arquivos mensais descompactados encontrados (2024 + 2025): {len(arquivos_csv)}")

if len(arquivos_csv) > 0:
    # 3. Leitura unificada de auditoria inicial (as-is)
    df_audit_v2 = spark.read \
        .option("header", "true") \
        .option("sep", ";") \
        .option("encoding", "ISO-8859-1") \
        .csv("/content/scr_v2/*.csv")

    print("=" * 70)
    print("📌 1. VOLUMETRIA TOTAL DE REGISTROS (2024-2025):")
    total_linhas = df_audit_v2.count()
    print(f"   Total de registros consolidados: {total_linhas:,}")
    print("=" * 70)

    print("\n📌 2. ESTRUTURA DE COLUNAS ENCONTRADAS:")
    print(f"   Total de colunas disponíveis: {len(df_audit_v2.columns)}")
    for idx, col_name in enumerate(df_audit_v2.columns, 1):
        print(f"   {idx:02d}. {col_name}")
    print("=" * 70)

    print("\n📌 3. VISUALIZAÇÃO DAS PRIMEIRAS LINHAS (HEAD):")
    df_audit_v2.show(5, truncate=False)
    print("=" * 70)

    print("\n📌 4. SCHEMA INICIAL (TIPAGEM BRUTA INFERIDA COMO STRING):")
    df_audit_v2.printSchema()
else:
    print("❌ Erro: Nenhum arquivo de dados foi extraído do repositório.")

⏳ Baixando repositórios oficiais scrdata_2024.zip e scrdata_2025.zip...
✅ Arquivos mensais descompactados encontrados (2024 + 2025): 24
📌 1. VOLUMETRIA TOTAL DE REGISTROS (2024-2025):
   Total de registros consolidados: 7,492,208

📌 2. ESTRUTURA DE COLUNAS ENCONTRADAS:
   Total de colunas disponíveis: 24
   01. data_base
   02. uf
   03. segmento
   04. cliente
   05. cnae_ocupacao
   06. porte
   07. modalidade
   08. submodalidade
   09. origem
   10. indexador
   11. numero_de_operacoes
   12. a_vencer_ate_90_dias
   13. a_vencer_de_91_ate_360_dias
   14. a_vencer_de_361_ate_1080_dias
   15. a_vencer_de_1081_ate_1800_dias
   16. a_vencer_de_1801_ate_5400_dias
   17. a_vencer_acima_de_5400_dias
   18. carteira_a_vencer
   19. vencido_de_15_ate_90_dias
   20. vencido_acima_de_90_dias
   21. carteira_vencida
   22. carteira_ativa
   23. carteira_inadimplencia
   24. ativo_problematico

📌 3. VISUALIZAÇÃO DAS PRIMEIRAS LINHAS (HEAD):
+----------+---+------------+-------+-----------------

### 2.1 Resumo dos Dados Auditados (Metadados Iniciais)

Após a consolidação e execução do script de auditoria sobre os repositórios oficiais da **Versão 2 do SCR.data** referentes ao biênio **2024–2025**, apuraram-se os seguintes metadados estruturais:

* **Fonte de Dados:** Banco Central do Brasil (BCB) / Portal de Dados Abertos  
* **Período Coberto:** 2024 e 2025 (24 arquivos mensais históricos)  
* **Total de Registros Consolidados:** 7.492.208 de linhas  
* **Total de Colunas:** 24 colunas  
* **Formato dos Dados:** Formato Largo (*Wide*) — pronto para vetorização direta de atributos sem necessidade de pivotagem.

Com a estrutura de colunas e os metadados iniciais devidamente identificados, a próxima etapa consiste na definição do esquema de dados (`StructType`) e na preparação dos atributos para a etapa de exploração e limpeza.

In [3]:
# =====================================================================
# ETAPA 3: INGESTÃO DEFINITIVA COM SCHEMA MANUAL (STRUCTTYPE)
# =====================================================================

from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql import functions as F

# ---------------------------------------------------------------------
# 3.1 Definição Manual do Schema (StructType - 24 Colunas)
# ---------------------------------------------------------------------
schema_scr = StructType([
    StructField("data_base", StringType(), True),
    StructField("uf", StringType(), True),
    StructField("segmento", StringType(), True),
    StructField("cliente", StringType(), True),
    StructField("cnae_ocupacao", StringType(), True),
    StructField("porte", StringType(), True),
    StructField("modalidade", StringType(), True),
    StructField("submodalidade", StringType(), True),
    StructField("origem", StringType(), True),
    StructField("indexador", StringType(), True),
    StructField("numero_de_operacoes", StringType(), True),
    StructField("a_vencer_ate_90_dias", StringType(), True),
    StructField("a_vencer_de_91_ate_360_dias", StringType(), True),
    StructField("a_vencer_de_361_ate_1080_dias", StringType(), True),
    StructField("a_vencer_de_1081_ate_1800_dias", StringType(), True),
    StructField("a_vencer_de_1801_ate_5400_dias", StringType(), True),
    StructField("a_vencer_acima_de_5400_dias", StringType(), True),
    StructField("carteira_a_vencer", StringType(), True),
    StructField("vencido_de_15_ate_90_dias", StringType(), True),
    StructField("vencido_acima_de_90_dias", StringType(), True),
    StructField("carteira_vencida", StringType(), True),
    StructField("carteira_ativa", StringType(), True),
    StructField("carteira_inadimplencia", StringType(), True),
    StructField("ativo_problematico", StringType(), True)
])

# ---------------------------------------------------------------------
# 3.2 Leitura Tipada dos Arquivos Consolidados (2024-2025)
# ---------------------------------------------------------------------
df_raw = spark.read \
    .option("header", "true") \
    .option("sep", ";") \
    .option("encoding", "ISO-8859-1") \
    .schema(schema_scr) \
    .csv("/content/scr_v2/*.csv")

# ---------------------------------------------------------------------
# 3.3 Conversão dos Campos Monetários
# ---------------------------------------------------------------------
# Mapeamento das 13 colunas financeiras que contêm valores decimais com vírgula
colunas_financeiras = [
    "a_vencer_ate_90_dias", "a_vencer_de_91_ate_360_dias",
    "a_vencer_de_361_ate_1080_dias", "a_vencer_de_1081_ate_1800_dias",
    "a_vencer_de_1801_ate_5400_dias", "a_vencer_acima_de_5400_dias",
    "carteira_a_vencer", "vencido_de_15_ate_90_dias",
    "vencido_acima_de_90_dias", "carteira_vencida",
    "carteira_ativa", "carteira_inadimplencia",
    "ativo_problematico"
]

df_ingested = df_raw

# Substituição da vírgula por ponto decimal e Cast para DoubleType
for col_name in colunas_financeiras:
    df_ingested = df_ingested.withColumn(
        col_name,
        F.regexp_replace(F.col(col_name), ",", ".").cast(DoubleType())
    )

# ---------------------------------------------------------------------
# 3.4 Validação do Schema Final e Amostra Tipada
# ---------------------------------------------------------------------
print("=" * 70)
print("✅ INGESTÃO E CONVERSÃO DE TIPOS CONCLUÍDAS COM SUCESSO!")
print("=" * 70)

print("\n📌 SCHEMA FINAL TIPADO (PYSPARK):")
df_ingested.printSchema()

print("\n📌 VISUALIZAÇÃO DOS DADOS MONETÁRIOS TIPADOS (DOUBLE):")
df_ingested.select(
    "data_base", "uf", "cliente", "carteira_ativa",
    "carteira_vencida", "carteira_inadimplencia", "ativo_problematico"
).show(5, truncate=False)

✅ INGESTÃO E CONVERSÃO DE TIPOS CONCLUÍDAS COM SUCESSO!

📌 SCHEMA FINAL TIPADO (PYSPARK):
root
 |-- data_base: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- cnae_ocupacao: string (nullable = true)
 |-- porte: string (nullable = true)
 |-- modalidade: string (nullable = true)
 |-- submodalidade: string (nullable = true)
 |-- origem: string (nullable = true)
 |-- indexador: string (nullable = true)
 |-- numero_de_operacoes: string (nullable = true)
 |-- a_vencer_ate_90_dias: double (nullable = true)
 |-- a_vencer_de_91_ate_360_dias: double (nullable = true)
 |-- a_vencer_de_361_ate_1080_dias: double (nullable = true)
 |-- a_vencer_de_1081_ate_1800_dias: double (nullable = true)
 |-- a_vencer_de_1801_ate_5400_dias: double (nullable = true)
 |-- a_vencer_acima_de_5400_dias: double (nullable = true)
 |-- carteira_a_vencer: double (nullable = true)
 |-- vencido_de_15_ate_90_dias: double (nullable

## 3. INGESTÃO DEFINITIVA E TRATAMENTO DE TIPOS

Nesta etapa, aplicou-se o contrato rígido de dados através da API `StructType` do PySpark, cobrindo manualmente todas as 24 colunas identificadas na base **SCR.data (Versão 2)**.

Como a base original armazena os valores numéricos em formato texto no padrão brasileiro (utilizando vírgula como separador decimal, ex: `"159911,55"`), realizou-se a sanitização dos atributos monetários via expressão regular (`regexp_replace`) e a posterior conversão explícita para o tipo `DoubleType`.

> **Nota Metodológica sobre o atributo `numero_de_operacoes`:**
> O campo `numero_de_operacoes` foi mantido temporariamente como `StringType` nesta fase de ingestão. A auditoria inicial revelou a existência de valores numéricos negativos (como `-1`), que costumam indicar códigos de exceção, dados ocultados por sigilo ou categorizações específicas do Banco Central. O campo será auditado detalhadamente na **Etapa 4 (Exploração e Limpeza)** antes de sua conversão final para tipo numérico (`IntegerType`/`DoubleType`), garantindo que valores mascarados não distorçam as análises estatísticas nem o agrupamento via **K-Means**.

Esta abordagem garante que o algoritmo de *Machine Learning* e as funções agregadoras do Spark operem com precisão numérica, mantendo o controle total sobre o ciclo de vida dos dados.

In [ ]:
# =====================================================================
# ETAPA 4: EXPLORAÇÃO E QUALIDADE DOS DADOS
# =====================================================================

from pyspark.sql import functions as F

# =====================================================================
# 4.1 AUDITORIA DO CAMPO numero_de_operacoes (FOCO: OCORRÊNCIAS DE -1)
# =====================================================================

print("--- 4.1.1 Diagnóstico Geral de numero_de_operacoes == -1 ---")
total_registros = df_ingested.count()
qtd_menos_um = df_ingested.filter(F.col("numero_de_operacoes") == -1).count()
pct_menos_um = (qtd_menos_um / total_registros) * 100

print(f"Total de registros na base: {total_registros:,}")
print(f"Registros com numero_de_operacoes == -1: {qtd_menos_um:,}")
print(f"Percentual representativo: {pct_menos_um:.4f}%\n")

print("--- 4.1.2 Distribuição do valor -1 por Segmento ---")
df_ingested.filter(F.col("numero_de_operacoes") == -1) \
    .groupBy("segmento") \
    .agg(
        F.count("*").alias("qtd_menos_1"),
        (F.count("*") / qtd_menos_um * 100).alias("pct_do_total_menos_1")
    ) \
    .orderBy(F.desc("qtd_menos_1")) \
    .show(truncate=False)

print("--- 4.1.3 Distribuição do valor -1 por Tipo de Cliente ---")
df_ingested.filter(F.col("numero_de_operacoes") == -1) \
    .groupBy("cliente") \
    .agg(
        F.count("*").alias("qtd_menos_1"),
        (F.count("*") / qtd_menos_um * 100).alias("pct_do_total_menos_1")
    ) \
    .orderBy(F.desc("qtd_menos_1")) \
    .show(truncate=False)

print("--- 4.1.4 Distribuição do valor -1 por Porte do Cliente ---")
df_ingested.filter(F.col("numero_de_operacoes") == -1) \
    .groupBy("porte") \
    .agg(
        F.count("*").alias("qtd_menos_1"),
        (F.count("*") / qtd_menos_um * 100).alias("pct_do_total_menos_1")
    ) \
    .orderBy(F.desc("qtd_menos_1")) \
    .show(truncate=False)

print("--- 4.1.5 Top 10 Modalidades com ocorrência do valor -1 ---")
df_ingested.filter(F.col("numero_de_operacoes") == -1) \
    .groupBy("modalidade") \
    .agg(
        F.count("*").alias("qtd_menos_1"),
        (F.count("*") / qtd_menos_um * 100).alias("pct_do_total_menos_1")
    ) \
    .orderBy(F.desc("qtd_menos_1")) \
    .show(10, truncate=False)

--- 4.1.1 Diagnóstico Geral de numero_de_operacoes == -1 ---
Total de registros na base: 7,492,208
Registros com numero_de_operacoes == -1: 2,013,553
Percentual representativo: 26.8753%

--- 4.1.2 Distribuição do valor -1 por Segmento ---
+--------------------------+-----------+--------------------+
|segmento                  |qtd_menos_1|pct_do_total_menos_1|
+--------------------------+-----------+--------------------+
|Banco                     |799507     |39.70628039093086   |
|Cooperativa               |543781     |27.006043545911133  |
|Financeira                |235110     |11.676375044510873  |
|Fintech                   |187534     |9.31358648121008    |
|Outros                    |86812      |4.311383906954522   |
|Desenvolvimento/Fomento   |81201      |4.03272225762123    |
|InstituiÃ§Ã£o de pagamento|55382      |2.7504614976610995  |
|Arrendamento              |24226      |1.2031468752002057  |
+--------------------------+-----------+--------------------+

--- 4.1.3 Distri

### 4.1 Conclusões da Auditoria do Campo `numero_de_operacoes`

A auditoria realizada sobre o atributo **`numero_de_operacoes`** identificou a presença do valor especial **`-1`** em **2.013.553 registros**, o que representa **26,88%** do volume total da base analisada (7.492.208 registros).

A investigação documental baseada na metodologia oficial do **SCR.data** (Banco Central do Brasil) indica que os dados agregados referentes à quantidade de operações estão sujeitos a mecanismos de proteção estatística e preservação do sigilo bancário. A documentação estabelece formalmente que, em células de agregação cujo número de operações seja igual ou inferior a 15, a informação divulgada passa por mascaramento, sendo originalmente representada pelo rótulo **`"<= 15"`**.

Embora o manual não declare de forma textual que os arquivos exportados em formato plano (*CSV/Dados Abertos*) convertem a string `"<="` no inteiro **`-1`**, a ampla e estruturada distribuição dessa ocorrência em todas as dimensões de negócio — segmentos institucionais, portes de tomadores, categorias de clientes (PF/PJ), modalidades operacionais e unidades da federação — sugere fortemente que o valor identificado atua como uma representação numérica do mecanismo de mascaramento por sigilo estatístico adotado na divulgação dos dados.

Diante do respaldo regulatório e metodológico obtido:

1. **Classificação:** O valor **`-1`** **não** será tratado como valor nulo, erro de ingestão ou inconsistência de qualidade dos dados.
2. **Tratamento na Etapa 4:** O atributo será **mantido estritamente sem alterações** nesta fase exploratória.
3. **Diretriz para a Etapa 5 (Engenharia de Atributos & Modelagem):** Sua aplicação e eventual transformação (como a criação de uma *flag* indicadora de sigilo estatístico ou tratamentos específicos de codificação) serão avaliadas de maneira criteriosa, garantindo que a presença da flag não introduza viés ou distorções nos algoritmos de aprendizado de máquina e no processo de clusterização.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType

# =====================================================================
# 4.2 LEVANTAMENTO DE VALORES NULOS POR COLUNA
# =====================================================================

print("--- 4.2 Levantamento de Nulos e Ausentes ---")

# Identifica o tipo de cada coluna para aplicar isnan apenas onde for numérico contínuo (Float/Double)
nulos_exprs = []
for col_name, col_type in df_ingested.dtypes:
    if col_type in ["double", "float"]:
        # Para tipos numéricos de ponto flutuante, verifica isNull e isnan
        nulos_exprs.append(
            F.sum(F.when(F.col(col_name).isNull() | F.isnan(F.col(col_name)), 1).otherwise(0)).alias(col_name)
        )
    else:
        # Para string, date, int e demais tipos, verifica apenas isNull
        nulos_exprs.append(
            F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias(col_name)
        )

# Executa a contagem agregada em uma única ação no Spark
df_nulos_count = df_ingested.select(nulos_exprs)

# Formata o resultado em tabela (Coluna | Nulos | %)
nulos_lista = []
nulos_dict = df_nulos_count.first().asDict()

for col_name, nulos_cnt in nulos_dict.items():
    pct = (nulos_cnt / total_registros) * 100
    nulos_lista.append((col_name, nulos_cnt, round(pct, 4)))

df_nulos_summary = spark.createDataFrame(nulos_lista, ["Coluna", "Nulos", "Percentual (%)"])
df_nulos_summary.orderBy(F.desc("Nulos")).show(len(df_ingested.columns), truncate=False)

--- 4.2 Levantamento de Nulos e Ausentes ---
+------------------------------+-----+--------------+
|Coluna                        |Nulos|Percentual (%)|
+------------------------------+-----+--------------+
|data_base                     |0    |0.0           |
|a_vencer_de_91_ate_360_dias   |0    |0.0           |
|uf                            |0    |0.0           |
|a_vencer_de_361_ate_1080_dias |0    |0.0           |
|segmento                      |0    |0.0           |
|a_vencer_de_1081_ate_1800_dias|0    |0.0           |
|cliente                       |0    |0.0           |
|a_vencer_de_1801_ate_5400_dias|0    |0.0           |
|cnae_ocupacao                 |0    |0.0           |
|a_vencer_acima_de_5400_dias   |0    |0.0           |
|porte                         |0    |0.0           |
|carteira_a_vencer             |0    |0.0           |
|modalidade                    |0    |0.0           |
|vencido_de_15_ate_90_dias     |0    |0.0           |
|submodalidade                 |0    

### 4.2 Conclusões do Levantamento de Valores Nulos

A verificação sistemática realizada em todas as 24 variáveis do dataset resultou em **0% de valores nulos ou ausentes (`NULL` / `NaN`)** em 100% dos 7.492.208 registros analisados.

#### Principais Observações Técnicas e Regulatórias

1. **Integridade da Carga:** Os processos de extração e ingestão na zona de *Staging/Data Lake* garantiram total aderência à estrutura original da base, sem perdas, truncamentos ou corrupção de atributos durante o processo de leitura e conversão dos dados.
2. **Tratamento de Categorias Indisponíveis:** Conforme identificado nas documentações metodológicas do **SCR.data**, situações em que determinadas características cadastrais não estão disponíveis são representadas por categorias explícitas, como `"Indisponível"`, e não por omissão de preenchimento[cite: 14, 15]. Dessa forma, a ausência de informação é tratada pelo próprio Banco Central como uma categoria válida de negócio, preservando a completude estrutural da base[cite: 13, 14, 15].
3. **Impactos no Pipeline Spark ML:** A inexistência de valores nulos elimina a necessidade de aplicação de técnicas de imputação estatística (como preenchimento por média, mediana ou moda via `Imputer`). Consequentemente, as etapas futuras de Engenharia de Atributos poderão concentrar-se exclusivamente na codificação (*StringIndexer* / *OneHotEncoder*), transformação e padronização das variáveis relevantes.
4. **Qualidade dos Dados:** Sob a perspectiva de completude, a base apresenta elevado nível de qualidade, característica compatível com sua origem regulatória e com os rigorosos processos de validação adotados pelo Banco Central do Brasil antes da divulgação pública dos dados[cite: 13].

Conclui-se que **não serão necessárias intervenções de limpeza relacionadas a valores ausentes**, permitindo que as próximas análises de qualidade concentrem-se na verificação de duplicidades, cardinalidade das variáveis categóricas e comportamento estatístico das variáveis financeiras.

In [ ]:
# =====================================================================
# 4.3 VERIFICAÇÃO DE DUPLICATAS
# =====================================================================

print("--- 4.3 Verificação de Registros Duplicados ---")

total_count = df_ingested.count()
distinct_count = df_ingested.dropDuplicates().count()
qtd_duplicatas = total_count - distinct_count

print(f"Total de registros na base: {total_count:,}")
print(f"Total de registros únicos:   {distinct_count:,}")
print(f"Quantidade de duplicatas:    {qtd_duplicatas:,}")
print(f"Percentual de duplicatas:    {(qtd_duplicatas / total_count) * 100:.4f}%")

--- 4.3 Verificação de Registros Duplicados ---
Total de registros na base: 7,492,208
Total de registros únicos:   7,492,208
Quantidade de duplicatas:    0
Percentual de duplicatas:    0.0000%


### 4.3 Conclusões da Verificação de Duplicatas

A análise de duplicidade foi realizada sobre os **7.492.208 registros** consolidados da base **SCR.data (Versão 2)**, considerando todas as 24 colunas disponíveis após a etapa de ingestão e padronização dos tipos de dados.

Os resultados demonstraram que o total de registros observados é idêntico ao total de registros únicos da base, evidenciando a **inexistência de duplicatas exatas** no conjunto de dados analisado (`0` duplicatas, **0,0000%**).

#### Principais Observações Técnicas e Regulatórias

1. **Unicidade dos Registros:** A ausência de registros duplicados demonstra que cada linha da base representa uma observação distinta dentro da estrutura agregada divulgada pelo Banco Central do Brasil, não havendo repetições integrais entre os registros analisados.
2. **Preservação da Integridade da Ingestão:** A inexistência de registros duplicados após a consolidação dos arquivos indica que o processo de ingestão preservou corretamente a granularidade dos dados e não introduziu repetições indevidas no conjunto analisado.
3. **Impacto na Etapa de Engenharia de Atributos e Modelagem:** Não foram identificados registros repetidos que pudessem provocar super-representação de observações, distorções em estatísticas descritivas ou vieses nos algoritmos de *Machine Learning*. Consequentemente, não serão necessárias etapas de deduplicação (`dropDuplicates()`) no PySpark ML, evitando o consumo desnecessário de memória e de operações de *shuffle* entre os nós do *cluster*.

A ausência simultânea de valores nulos (conforme 4.2) e de registros duplicados exatos reforça a elevada qualidade estrutural da base divulgada pelo Banco Central do Brasil. Conclui-se que nenhuma ação de remoção, deduplicação ou ajuste corretivo faz-se necessária nesta etapa, permitindo o avanço imediato para a análise de cardinalidade das variáveis categóricas.

In [ ]:
# =====================================================================
# 4.4 CARDINALIDADE DAS VARIÁVEIS CATEGÓRICAS
# =====================================================================

print("--- 4.4 Contagem de Valores Únicos (Cardinalidade) ---")

cols_categoricas = [
    "uf", "segmento", "cliente", "porte",
    "modalidade", "submodalidade", "origem", "indexador"
]

cardinalidade_lista = []
for c in cols_categoricas:
    if c in df_ingested.columns:
        unicos = df_ingested.select(c).distinct().count()
        cardinalidade_lista.append((c, unicos))

df_cardinalidade = spark.createDataFrame(cardinalidade_lista, ["Variavel_Categorica", "Valores_Distintos"])
df_cardinalidade.orderBy(F.desc("Valores_Distintos")).show(truncate=False)

--- 4.4 Contagem de Valores Únicos (Cardinalidade) ---
+-------------------+-----------------+
|Variavel_Categorica|Valores_Distintos|
+-------------------+-----------------+
|submodalidade      |56               |
|uf                 |27               |
|modalidade         |13               |
|porte              |13               |
|segmento           |8                |
|indexador          |6                |
|cliente            |2                |
|origem             |2                |
+-------------------+-----------------+



### 4.4 Conclusões da Análise de Cardinalidade das Variáveis Categóricas

A aferição de cardinalidade mapeou a quantidade de valores distintos para as 8 variáveis categóricas primárias da base **SCR.data (Versão 2)**, totalizando 7.492.208 registros.

| Variável Categórica | Valores Distintos | Descrição e Comportamento de Negócio |
| :--- | :---: | :--- |
| **submodalidade** | **56** | Elevada granularidade operacional definida pelo BCB (Anexo 3 do Doc 3040). |
| **uf** | **27** | Abrangência geográfica total cobrindo as 26 Unidades da Federação mais o Distrito Federal. |
| **modalidade** | **13** | Agrupamentos macro de produtos de crédito (ex: *Empréstimos*, *Financiamentos*, *Habitacional*). |
| **porte** | **13** | Faixas de rendimento/faturamento cobrindo PF (8 faixas + Indisponível/Sem rendimento) e PJ (4 portes + Indisponível). |
| **segmento** | **8** | Agrupamentos de instituições autorizadas (*Banco*, *Cooperativa*, *Financeira*, *Fintech*, *IP*, etc.). |
| **indexador** | **6** | Taxas referenciais de correção (*Prefixado*, *Pós-fixado*, *Flutuantes*, *Índices de Preços*, *TCR/TRFC*, *Outros*). |
| **cliente** | **2** | Segmentação binária de personalidade jurídica (`PF` - Pessoa Física / `PJ` - Pessoa Jurídica). |
| **origem** | **2** | Destinação regulatória dos recursos (`Sem destinação específica` - Recursos Livres / `Com destinação específica` - Recursos Direcionados). |

---

#### Principais Observações Técnicas e Diretrizes para o Spark ML

1. **Equilíbrio entre Riqueza Semântica e Complexidade Computacional:** A variável `submodalidade` apresenta o maior nível de detalhamento operacional (56 categorias), fornecendo capacidade de diferenciação entre produtos e operações de crédito sem introduzir cardinalidade excessiva. Essa característica favorece a formação de *clusters* mais informativos durante a etapa de modelagem.
2. **Baixa e Média Cardinalidade Predominantes:** Todas as variáveis categóricas apresentaram baixa a moderada cardinalidade ($\le 56$ valores distintos), o que representa uma importante vantagem computacional para o processamento distribuído em ambiente PySpark.
3. **Estratégia de Codificação na Etapa 5 (Engenharia de Atributos):**
   * **`StringIndexer`:** Todas as 8 variáveis categóricas podem ser indexadas numericamente via `StringIndexer` sem risco de gargalo de memória (*out-of-memory*) no nó *driver*.
   * **`OneHotEncoder` (OHE):** Como a cardinalidade máxima observada é de apenas 56 categorias (`submodalidade`), a expansão do espaço de atributos permanece controlada e adequada para utilização em algoritmos de agrupamento (como o **K-Means**), sem expectativa de explosão dimensional significativa.
4. **Validação de Domínio:** O mapeamento exato de **27 UFs** e **2 tipos de cliente (PF/PJ)** confirma a consistência dos dados recebidos e valida a integridade do processo de ingestão e tipagem realizado na Etapa 3.

Conclui-se que o conjunto de atributos categóricos encontra-se **adequadamente mapeado, validado e estruturalmente preparado** para as etapas de codificação e integração ao Pipeline Spark ML.

In [4]:
# =====================================================================
# 4.5 ESTATÍSTICAS DESCRITIVAS DAS VARIÁVEIS FINANCEIRAS
# =====================================================================

print("--- 4.5 Resumo Estatístico dos Atributos Financeiros ---")

cols_financeiras = [
    "carteira_a_vencer",
    "carteira_vencida",
    "carteira_ativa",
    "carteira_inadimplencia",
    "ativo_problematico"
]

# Exibe métricas de centralidade e dispersão (Count, Mean, Stddev, Min, Max)
df_ingested.select(cols_financeiras).describe().show(truncate=False)

print("--- 4.5.1 Cálculo de Percentis (Quartis e Mediana) ---")
# Percentis 25%, 50% (Mediana), 75% e 99% (para avaliação de cauda)
percentis = [0.25, 0.50, 0.75, 0.99]

for col_fin in cols_financeiras:
    quantiles = df_ingested.approxQuantile(col_fin, percentis, 0.01)
    print(f"Atributo: {col_fin}")
    print(f"  P25: {quantiles[0]:,.2f} | P50 (Mediana): {quantiles[1]:,.2f} | P75: {quantiles[2]:,.2f} | P99: {quantiles[3]:,.2f}\n")

--- 4.5 Resumo Estatístico dos Atributos Financeiros ---
+-------+--------------------+-----------------+--------------------+----------------------+--------------------+
|summary|carteira_a_vencer   |carteira_vencida |carteira_ativa      |carteira_inadimplencia|ativo_problematico  |
+-------+--------------------+-----------------+--------------------+----------------------+--------------------+
|count  |7492208             |7492208          |7492208             |7492208               |7492208             |
|mean   |2.0692495256442748E7|542479.1764975407|2.1234974432528105E7|741870.8002657779     |1521752.8051415102  |
|stddev |2.9168237010946393E8|6682933.362825686|2.930828630664922E8 |7901727.555839506     |2.2234576513971224E7|
|min    |0.0                 |0.0              |1.01                |0.0                   |0.0                 |
|max    |1.0654345402057E11  |4.63216094975E9  |1.0654345402057E11  |4.45422102744E9       |8.95825393668E9     |
+-------+--------------------+-

### 4.5 Conclusões do Resumo Estatístico das Variáveis Financeiras

A análise das propriedades estatísticas e de dispersão das 5 variáveis monetárias da base **SCR.data (Versão 2)** revela aspectos fundamentais sobre a distribuição dos saldos de crédito no Sistema Financeiro Nacional.

#### Tabela Consolidada de Métricas Estatísticas (Valores em R$)

| Atributo Financeiro | Média | Desvio Padrão ($\sigma$) | P25 (1º Quartil) | P50 (Mediana) | P75 (3º Quartil) | P99 (Percentil 99) | Mínimo | Máximo |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **`carteira_a_vencer`** | R$ 20.692.495,26 | R$ 291.682.370,11 | R$ 13.993,89 | R$ 181.409,22 | R$ 1.802.977,87 | R$ 106.543.454.020,57 | R$ 0,00 | R$ 106,54 Bi |
| **`carteira_vencida`** | R$ 542.479,18 | R$ 6.682.933,36 | R$ 0,00 | R$ 773,10 | R$ 42.377,79 | R$ 4.632.160.949,75 | R$ 0,00 | R$ 4,63 Bi |
| **`carteira_ativa`** | R$ 21.234.974,43 | R$ 293.082.863,07 | R$ 26.443,77 | R$ 242.306,21 | R$ 2.098.130,99 | R$ 106.543.454.020,57 | R$ 1,01 | R$ 106,54 Bi |
| **`carteira_inadimplencia`** | R$ 741.870,80 | R$ 7.901.727,56 | R$ 0,00 | R$ 0,00 | R$ 44.311,40 | R$ 4.454.221.027,44 | R$ 0,00 | R$ 4,45 Bi |
| **`ativo_problematico`** | R$ 1.521.752,81 | R$ 22.234.576,51 | R$ 0,00 | R$ 2.535,61 | R$ 122.070,47 | R$ 8.958.253.936,68 | R$ 0,00 | R$ 8,96 Bi |

---

#### Principais Observações Analíticas e de Negócio

1. **Extrema Assimetria à Direita e Concentração de Crédito:**
   * A constatação mais relevante da análise reside na acentuada disparidade entre média e mediana: na `carteira_ativa`, a média (R$ 21,23 milhões) é **87,64 vezes superior à mediana** (R$ 242,3 mil).
   * Essa assimetria extrema reflete a elevada concentração do mercado de crédito brasileiro, onde a imensa maioria dos registros refere-se a volumes menores (Pessoas Físicas e Micro/Pequenas Empresas), enquanto a cauda longa da distribuição é tracionada por grandes exposições corporativas e conglomerados financeiros atingindo patamares bilionários em agregações específicas.

2. **Comportamento das Carteiras de Inadimplência e Ativo Problemático:**
   * A **`carteira_inadimplencia`** apresenta **P50 (mediana) igual a R$ 0,00**, indicando que em mais da metade das combinações agregadas da base não há registro de atrasos superiores a 90 dias. A inadimplência manifesta-se de forma mais expressiva a partir do 3º quartil (P75: R$ 44.311,40).
   * O **`ativo_problematico`** exibe valores consistentemente superiores aos da `carteira_inadimplencia` (média de R$ 1,52 milhão contra R$ 741,8 mil). Essa diferença é coerente com os conceitos regulatórios adotados pelo Banco Central (Resoluções CMN 4.557 e 4.966), visto que a métrica de ativo problemático abrange não apenas a inadimplência estrita ($\ge 90$ dias), mas também reestruturações de dívida, operações renegociadas e ativos com indicativos de deterioração de risco (níveis E a H).

3. **Validação Operacional da `carteira_ativa`:**
   * O valor mínimo de **R$ 1,01** confirma a ausência de registros com valor zero na carteira total ativa, atestando a consistência dos dados agrupados pelo Banco Central.

---

#### Decisão Metodológica e Implicações para a Etapa 5 (Engenharia de Atributos)

Os resultados evidenciam distribuições fortemente assimétricas e com elevada concentração em poucos registros de grande magnitude. Dessa forma, as variáveis financeiras serão candidatas naturais à transformação logarítmica ($\log(1+x)$ / `log1p()`) e posterior padronização (via `StandardScaler` ou `RobustScaler`) na etapa de Engenharia de Atributos, antes da aplicação dos algoritmos de clusterização no PySpark ML.

In [ ]:
# =====================================================================
# 4.6 ANÁLISE DE DISTRIBUIÇÃO E ASSIMETRIA (OUTLIERS)
# =====================================================================

print("--- 4.6 Avaliação de Assimetria (Skewness) das Variáveis Financeiras ---")

skew_exprs = [F.skewness(c).alias(f"skew_{c}") for c in cols_financeiras]
df_skew = df_ingested.select(skew_exprs)
df_skew.show(truncate=False)

print("Nota de Interpretação:")
print("- Skewness próximo de 0: Distribuição Simétrica")
print("- Skewness > 1: Forte assimetria à direita (Requer transformação logarítmica / StandardScales no Spark ML)")

--- 4.6 Avaliação de Assimetria (Skewness) das Variáveis Financeiras ---
+----------------------+---------------------+-------------------+---------------------------+-----------------------+
|skew_carteira_a_vencer|skew_carteira_vencida|skew_carteira_ativa|skew_carteira_inadimplencia|skew_ativo_problematico|
+----------------------+---------------------+-------------------+---------------------------+-----------------------+
|89.17917318757033     |178.89866518583315   |88.39997069046001  |105.70470692089646         |110.344504071841       |
+----------------------+---------------------+-------------------+---------------------------+-----------------------+

Nota de Interpretação:
- Skewness próximo de 0: Distribuição Simétrica
- Skewness > 1: Forte assimetria à direita (Requer transformação logarítmica / StandardScales no Spark ML)


### 4.6 Conclusões da Análise de Assimetria (*Skewness*) e Outliers

A aferição dos coeficientes de assimetria (*skewness*) sobre as 5 variáveis monetárias da base **SCR.data (Versão 2)** quantifica a extrema cauda longa à direita observada no resumo estatístico (Subetapa 4.5).

#### Tabela Consolidada dos Coeficientes de Assimetria

| Atributo Financeiro | Coeficiente de Assimetria (*Skewness*) | Classificação Teórica | Raciocínio de Negócio / Causa Raiz |
| :--- | :---: | :--- | :--- |
| **`carteira_a_vencer`** | **89,18** | Extrema Assimetria Positiva ($\gg 1$) | Concentração do volume de crédito corporativo a vencer em grandes grupos econômicos. |
| **`carteira_vencida`** | **178,90** | Maior Assimetria da Base ($\gg 1$) | Efeito combinado de alta pulverização de inadimplência em PF/PMEs com eventos pontuais de grandíssimo porte. |
| **`carteira_ativa`** | **88,40** | Extrema Assimetria Positiva ($\gg 1$) | Reflexo direto da estrutura da carteira a vencer (que compõe a maior parcela da carteira ativa). |
| **`carteira_inadimplencia`** | **105,70** | Extrema Assimetria Positiva ($\gg 1$) | Concentração acentuada do saldo inadimplente $\ge 90$ dias em agregados específicos. |
| **`ativo_problematico`** | **110,34** | Extrema Assimetria Positiva ($\gg 1$) | Reestruturações e operações de risco elevado (níveis E a H) concentradas em grandes tomadores. |

---

#### Principais Observações Técnicas e de Modelagem

1. **Intensidade da Assimetria Positiva:**
   * Na teoria estatística, coeficientes de *skewness* superiores a $1{,}0$ indicam forte assimetria à direita. Os valores obtidos entre **88,40 e 178,90** confirmam uma distribuição severamente inclinada, na qual a imensa maioria dos registros agrupa-se em patamares modestos, enquanto uma fração mínima de observações se estende até a casa dos bilhões de reais.

2. **Tratamento de Outliers (Visão Preditiva x Agrupamento Regulatório):**
   * Em fluxos tradicionais de *Machine Learning*, observações tão afastadas seriam tratadas como *outliers* ruidosos e eventualmente truncadas ou removidas.
   * No entanto, no contexto do **SCR/BACEN**, esses valores bilionários não constituem erros ou anomalias de medição, mas sim **operações financeiras legítimas de alta relevância sistêmica**. A remoção ou truncamento (*clipping*) desses dados acarretaria perda substancial da representatividade do mercado de crédito.

3. **Justificativa Matemática Definitiva para a Transformação Logarítmica:**
   * A aplicação do algoritmo **K-Means** baseia-se no cálculo da Distância Euclidiana ($L_2$). Sem tratamento prévio, a atualização dos centroides seria completamente dominada pelos registros de grande magnitude, resultando em agrupamentos excessivamente influenciados por um pequeno conjunto de registros de grande magnitude e empurrando a esmagadora maioria da base para um único agrupamento residual.
   * A transformação $\log(1+x)$ (`log1p()`) comprime a escala de variação de várias ordens de grandeza para uma amplitude linear restrita, reduzindo a assimetria para patamares adequados e permitindo que o algoritmo diferencie os perfis intermediários de crédito.

---

### Encerrando a ETAPA 4: Síntese e Próximos Passos

A **ETAPA 4 (Exploração e Qualidade dos Dados)** é finalizada com um diagnóstico completo e rigoroso da base **SCR.data**:

* **4.1 Auditoria do `-1`:** Confirmado como código regulatório de mascaramento estatístico ($\le 15$ operações), devendo ser preservado sem alterações.
* **4.2 Nulos e Ausentes:** **0% de nulos** em 100% dos 7.492.208 registros (dispensando etapas de imputação).
* **4.3 Duplicatas:** **0 duplicatas exatas** (integridade da carga e unicidade dos registros validadas).
* **4.4 Cardinalidade:** Atributos categóricos apresentando cardinalidade moderada ($\le 56$), adequados para `StringIndexer` e `OneHotEncoder`.
* **4.5 & 4.6 Distribuição Financeira:** Comprovada extrema assimetria à direita ($\text{Média}/\text{Mediana} = 87{,}64$ e $\text{Skewness} > 88$), respaldando a estratégia de transformação logarítmica e padronização.

### ETAPA 5: ENGENHARIA DE ATRIBUTOS (*Feature Engineering*)

Com a base auditada e validada, o trabalho prossegue para a **ETAPA 5**, na qual será estruturado o Pipeline oficial do PySpark ML:

1. **Criação de Atributos Derivados:** *Flags* operacionais (ex.: indicadoras de sigilo/`-1`) e razões financeiras (como proporção da carteira inadimplida e ativo problemático em relação à carteira ativa).
2. **Transformação Logarítmica das Colunas Monetárias:** Aplicação de `F.log1p()` sobre os 5 atributos financeiros.
3. **Indexação e Vetorização de Categorias:** Encadeamento de `StringIndexer` e `OneHotEncoder` sobre as 8 variáveis categóricas.
4. **Padronização e Montagem do Vetor Final:** Aplicação de `VectorAssembler` e `StandardScaler` para consolidação da coluna `features` destinada ao algoritmo K-Means.

In [5]:
# =====================================================================
# ETAPA 5: ENGENHARIA DE ATRIBUTOS
# =====================================================================

from pyspark.sql import functions as F

print("--- 5.1 & 5.2 Construção dos Atributos Derivados e Transformações Logarítmicas (Ajustado) ---")

# 1. Indicador binário de sigilo estatístico (numero_de_operacoes == -1)
# A coluna original numero_de_operacoes é preservada intacta sem imputações arbitrárias
df_feat = df_ingested.withColumn(
    "flg_sigilo_operacoes",
    F.when(F.col("numero_de_operacoes") == -1, 1).otherwise(0)
)

# 2. Criação de Razões Financeiras (Ratios) com proteção contra divisão por zero
df_feat = df_feat.withColumn(
    "pct_inadimplencia",
    F.when(F.col("carteira_ativa") > 0, F.col("carteira_inadimplencia") / F.col("carteira_ativa")).otherwise(0.0)
).withColumn(
    "pct_ativo_problematico",
    F.when(F.col("carteira_ativa") > 0, F.col("ativo_problematico") / F.col("carteira_ativa")).otherwise(0.0)
)

# 3. Transformação log1p() das variáveis financeiras
cols_financeiras_brutas = [
    "carteira_a_vencer",
    "carteira_vencida",
    "carteira_ativa",
    "carteira_inadimplencia",
    "ativo_problematico"
]

for col_fin in cols_financeiras_brutas:
    df_feat = df_feat.withColumn(f"log_{col_fin}", F.log1p(F.col(col_fin)))

# Exibe confirmação das colunas criadas
cols_preview = [
    "flg_sigilo_operacoes",
    "pct_inadimplencia",
    "pct_ativo_problematico",
    "log_carteira_ativa",
    "log_carteira_inadimplencia"
]
df_feat.select(cols_preview).show(5, truncate=False)

--- 5.1 & 5.2 Construção dos Atributos Derivados e Transformações Logarítmicas (Ajustado) ---
+--------------------+-------------------+----------------------+------------------+--------------------------+
|flg_sigilo_operacoes|pct_inadimplencia  |pct_ativo_problematico|log_carteira_ativa|log_carteira_inadimplencia|
+--------------------+-------------------+----------------------+------------------+--------------------------+
|1                   |0.0                |0.0                   |13.996468418680593|0.0                       |
|1                   |0.0                |0.0                   |14.87040568599365 |0.0                       |
|1                   |0.0                |0.0                   |12.417274497732935|0.0                       |
|1                   |0.9878649629753999 |0.9878649629753999    |8.147710867715643 |8.1355051549675           |
|0                   |0.01662493923154142|0.039382571627051256  |14.874945946119315|10.778115102858067        |
+---------

### 5.1 & 5.2 Validação dos Atributos Derivados e Transformações Logarítmicas

A amostra dos registros processados valida a execução bem-sucedida das transformações numéricas e de negócio:

1. **Preservação de Sigilo:** A variável `flg_sigilo_operacoes` capturou corretamente as instâncias sujeitas ao mascaramento estatístico do Banco Central (`numero_de_operacoes == -1`), isolando a restrição de sigilo em uma dimensão binária sem corromper o dado quantitativo.
2. **Consistência dos *Ratios* de Risco:** As colunas `pct_inadimplencia` e `pct_ativo_problematico` demonstraram comportamento coerente com as regras do SCR, variando de $0,0$ a $0,9878$, permitindo a comparação relativa de carteiras de diferentes portes.
3. **Efetividade do `log1p`:** As variáveis monetárias em escala logarítmica apresentaram compressão adequada (com valores de `log_carteira_ativa` situados entre $8,14$ e $14,87$), com preservação exata dos valores nulos/zerados ($0,0 \to 0,0$).

### 5.3 a 5.6 Construção e Ajuste do Pipeline Spark ML

A Engenharia de Atributos foi concluída com o encadeamento dos estimadores e transformadores no **Spark ML Pipeline**, garantindo reprodutibilidade estrita, prevenção de *data leakage* e eficiência de memória na execução distribuída.

#### Matriz de Componentes do Pipeline

| Estágio | Componente ML | Entrada (`inputCol`) | Saída (`outputCol`) | Justificativa de Arquitetura & Otimização |
| :--- | :--- | :--- | :--- | :--- |
| **1** | `StringIndexer` (x8) | 8 cols categóricas | `*_indexed` | Indexação de rótulos categóricos em índices numéricos (`handleInvalid="keep"`). |
| **2** | `OneHotEncoder` (x8) | `*_indexed` | `*_ohe` | Conversão dos índices em vetores esparsos binários para evitar falsa ordenação ordinal. |
| **3** | `VectorAssembler` | Numéricas + `*_ohe` | `features_unscaled` | Consolidação de atributos numéricos, *ratios*, *flags* e vetores *OHE* em um único vetor. |
| **4** | `StandardScaler` | `features_unscaled` | `features` | **Otimização Distribuída:** Configurado com `withMean=False` e `withStd=True` para escalar a variância sem forçar a densificação de matrizes esparsas, prevenindo gargalos de memória (*OOM*) no cluster. |

---

#### Decisões de Governança e Impacto Metodológico

1. **Preservação do Rigor de Sigilo Estatístico:**
   * A *flag* binária `flg_sigilo_operacoes` converteu a limitação regulatória do mascaramento ($\le 15$ operações) em um vetor numérico utilizável pelo modelo ($0$ = quantidade divulgada, $1$ = restrição de sigilo), sem que fossem inventados valores arbitrários para a coluna `numero_de_operacoes`.

2. **Riqueza Preditiva das Razões Financeiras (*Ratios*):**
   * As variáveis `pct_inadimplencia` e `pct_ativo_problematico` normalizaram o risco de crédito na faixa $[0.0, 1.0]$, permitindo que instituições financeiras e carteiras de portes discrepantes (de milhares a bilhões de reais) sejam comparadas de forma homogênea no espaço vetorial.

3. **Adequação do Logaritmo e Escalonamento à Distância Euclidiana:**
   * A transformação $\log(1+x)$ suprimiu a assimetria desproporcional ($\text{Skewness} > 88$), enquanto o `StandardScaler` padronizou as variâncias. Dessa forma, o algoritmo K-Means operará sobre dimensões equilibradas, evitando a dominância de uma única variável sobre as distâncias de centroide.

In [6]:
# =====================================================================
# 5.3 A 5.6 CONSTRUÇÃO E AJUSTE DO PIPELINE SPARK ML
# =====================================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

print("--- 5.3 a 5.6 Execução do Pipeline Spark ML Otimizado ---")

# 1. Mapeamento das 8 variáveis categóricas
cols_categoricas = [
    "uf", "segmento", "cliente", "porte",
    "modalidade", "submodalidade", "origem", "indexador"
]

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    for col in cols_categoricas
]

encoders = [
    OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_ohe")
    for col in cols_categoricas
]

# 2. Definição do conjunto de variáveis numéricas e derivadas
cols_numericas_feat = [
    "log_carteira_a_vencer",
    "log_carteira_vencida",
    "log_carteira_ativa",
    "log_carteira_inadimplencia",
    "log_ativo_problematico",
    "pct_inadimplencia",
    "pct_ativo_problematico",
    "flg_sigilo_operacoes"
]

# Lista consolidada de entradas para o VectorAssembler
cols_ohe = [f"{col}_ohe" for col in cols_categoricas]
all_features = cols_numericas_feat + cols_ohe

# 3. VectorAssembler
assembler = VectorAssembler(
    inputCols=all_features,
    outputCol="features_unscaled",
    handleInvalid="skip"
)

# 4. StandardScaler Otimizado (Preserva Esparsidade: withMean=False)
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withMean=False,  # Evita a densificação desnecessária do vetor na memória
    withStd=True
)

# 5. Construção e Fit do Pipeline
pipeline_stages = indexers + encoders + [assembler, scaler]
pipeline = Pipeline(stages=pipeline_stages)

print("Executando fit() e transform() no cluster Spark...")
pipeline_model = pipeline.fit(df_feat)
df_prepared = pipeline_model.transform(df_feat)

# =====================================================================
# INSPEÇÃO E DOCUMENTAÇÃO DA DIMENSÃO FINAL
# =====================================================================

first_vector = df_prepared.select("features").first()[0]
print("\n" + "="*60)
print(f" Dimensão Total do Vetor 'features': {first_vector.size} posições")
print(f" Tipo de Representação Vetorial:    {type(first_vector).__name__}")
print("="*60)

df_prepared.select("features").show(2, truncate=False)

--- 5.3 a 5.6 Execução do Pipeline Spark ML Otimizado ---
Executando fit() e transform() no cluster Spark...

 Dimensão Total do Vetor 'features': 135 posições
 Tipo de Representação Vetorial:    SparseVector
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                                                                                                                |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|(135,[0,2,7,32,42,44,48,65,104,127,12

### Encerramento da ETAPA 5 — Engenharia de Atributos

A **Engenharia de Atributos** foi concluída com sucesso por meio da criação de atributos derivados (como *flags* de sigilo estatístico e *ratios* de risco de inadimplência e ativo problemático), transformação logarítmica $\log(1+x)$ das variáveis financeiras, codificação das 8 variáveis categóricas via `StringIndexer` e `OneHotEncoder`, e encadeamento no **Pipeline Spark ML**.

O processo resultou em um vetor final de **135 atributos** representados no formato `SparseVector`, com escalonamento via `StandardScaler(withMean=False)` — preservando a eficiência computacional e a gestão de memória RAM no *cluster* mesmo para um conjunto de dados contendo **7,49 milhões de registros**.

A coluna `features` produzida pelo Pipeline encontra-se plenamente validada e pronta para utilização em algoritmos de aprendizado não supervisionado, permitindo o avanço para a **ETAPA 6 – Modelagem com K-Means**.

In [7]:
print(df_prepared.count())
print(df_prepared.select("features").count())

7492208
7492208


### 5.6 Validação de Integridade e Completude da Coluna `features`

A auditoria de contagem sobre o DataFrame preparado (`df_prepared`) valida a integridade do Pipeline Spark ML:

* **Contagem Total da Base (`df_prepared.count()`):** 7.492.208 registros
* **Contagem da Coluna Vetorial (`select("features").count()`):** 7.492.208 registros

#### Diagnóstico Técnico

1. **Ausência de Descarte por Nulos (`handleInvalid="skip"`):** O `VectorAssembler` manteve a totalidade da base intacta, confirmando que a etapa de tratamento prévio garantiu 0% de nulos em todas as colunas de entrada.
2. **Consistência Dimensional:** A coluna `features` foi gerada para 100% dos registros no formato `SparseVector` (135 posições), garantindo a paridade entre a base bruta e o espaço de atributos pronto para a modelagem.

### ETAPA 6: MODELAGEM COM K-MEANS

A busca pela quantidade ideal de agrupamentos ($K$) é realizada por meio da avaliação combinada do **Método do Cotovelo (*Elbow Method* via WSSSE/Inércia)** e do **Coeficiente de Silhueta (*Silhouette Score*)**, com validação orientada à interpretabilidade regulatória e de negócio do SFN.

#### Diretrizes de Otimização no PySpark ML

1. **Persistência em Cache (`MEMORY_AND_DISK`):**
   * O DataFrame pós-pipeline `df_prepared` (7.492.208 registros e 135 atributos esparsos) é persistido no cluster antes do loop de iterabilidade.
   * A materialização prévia via `count()` garante que a leitura dos vetores ocorra diretamente da memória/disco a cada iteração de $K$, otimizando a taxa de transferência de I/O.

2. **Parametrização do Estimador `KMeans`:**
   * **`initMode="k-means||"`:** Implementação paralela e distribuída do algoritmo $K\text{-Means}++$, assegurando a escolha de centroides iniciais distantes entre si e acelerando a convergência.
   * **`maxIter=50` e `tol=1e-4`:** Critérios estritos de parada para garantir estabilidade numérica na posição final dos centroides.
   * **`seed=42`:** Garantia de reprodutibilidade estrita dos experimentos.

---

#### Critérios Formais de Decisão para Escolha do $K$ Final

| Critério Analítico | Métrica Utilizada | Regra de Avaliação |
| :--- | :--- | :--- |
| **Separabilidade e Coesão** | *Silhouette Score* (via `ClusteringEvaluator`) | Busca do valor máximo no intervalo $[2, 8]$. Faixas de referência: $>0,70$ (Excelente), $0,50\text{--}0,70$ (Muito Bom), $0,25\text{--}0,50$ (Razoável). |
| **Ganho de Variância Explicada** | WSSSE (*Within Set Sum of Squared Errors*) | Identificação do ponto de inflexão ("cotovelo") onde reduções adicionais de inércia apresentam retornos decrescentes. |
| **Interpretabilidade de Negócio** | Parcimônia e Relevância Regulatória | Preferência pelo menor número de *clusters* que mantenha separabilidade clara entre carteiras de crédito, perfis de risco e modalidades operacionais. |

In [9]:
# =====================================================================
# ETAPA 6: MODELAGEM COM K-MEANS
# Subetapa 6.1: Avaliação do Número Ótimo de Clusters (K)
# =====================================================================

from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

print("--- 6.1 Persistência Otimizada e Avaliação do Número Ótimo de Clusters (K) ---")

# 1. Isolamento da coluna vetorial e persistência em DISK_ONLY (Previne Java Heap Space OOM)
df_prepared_features = df_prepared.select("features")
df_prepared_features.persist(StorageLevel.DISK_ONLY)

total_registros = df_prepared_features.count()
print(f"Base otimizada e materializada em disco! Total de registros: {total_registros:,}\n")

# 2. Configuração do Avaliador de Silhueta
evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

# 3. Loop de avaliação do K-Means de K = 2 a K = 8
k_values = range(2, 9)
resultados_k = []

for k in k_values:
    t0 = time.time()

    # Instanciação com k-means|| e semente fixa
    kmeans = KMeans(
        k=k,
        featuresCol="features",
        predictionCol="prediction",
        seed=42,
        initMode="k-means||",
        maxIter=50,
        tol=1e-4
    )

    model = kmeans.fit(df_prepared_features)

    # Métricas de inércia e silhueta
    wssse = model.summary.trainingCost
    predictions = model.transform(df_prepared_features)
    silhouette = evaluator.evaluate(predictions)

    tempo_exec = time.time() - t0

    resultados_k.append((k, wssse, silhouette, tempo_exec))
    print(f"K = {k:02d} | WSSSE (Inércia): {wssse:20.2f} | Silhouette: {silhouette:.4f} | Tempo: {tempo_exec:.1f}s")

# 4. Tabela Consolidada de Resultados
df_res_k = spark.createDataFrame(resultados_k, ["K", "WSSSE", "Silhouette_Score", "Tempo_Segundos"])

print("\n" + "="*70)
print("--- Resultados Ordenados por K ---")
print("="*70)
df_res_k.orderBy("K").show(truncate=False)

print("\n" + "="*70)
print("--- Resultados Ordenados por Silhouette Score (Decrescente) ---")
print("="*70)
df_res_k.orderBy(F.col("Silhouette_Score").desc()).show(truncate=False)

--- 6.1 Persistência Otimizada e Avaliação do Número Ótimo de Clusters (K) ---
Base otimizada e materializada em disco! Total de registros: 7,492,208

K = 02 | WSSSE (Inércia):         986446735.83 | Silhouette: 0.1427 | Tempo: 818.6s
K = 03 | WSSSE (Inércia):         970251407.11 | Silhouette: 0.0880 | Tempo: 539.6s
K = 04 | WSSSE (Inércia):         953681761.36 | Silhouette: -0.0134 | Tempo: 1260.2s
K = 05 | WSSSE (Inércia):         939374167.46 | Silhouette: 0.0193 | Tempo: 1235.8s
K = 06 | WSSSE (Inércia):         921773975.79 | Silhouette: 0.0396 | Tempo: 1165.3s
K = 07 | WSSSE (Inércia):         928759320.59 | Silhouette: 0.0142 | Tempo: 1553.8s
K = 08 | WSSSE (Inércia):         902811264.08 | Silhouette: 0.0177 | Tempo: 860.5s

--- Resultados Ordenados por K ---
+---+-------------------+---------------------+------------------+
|K  |WSSSE              |Silhouette_Score     |Tempo_Segundos    |
+---+-------------------+---------------------+------------------+
|2  |9.864467358335

### 6.2 Justificativa Estatística e Seleção do Modelo Definitivo ($K = 2$)

A análise conjunta do Coeficiente de Silhueta (*Silhouette Score*) e do *Within Set Sum of Squared Errors* (WSSSE / Inércia) fundamenta a escolha do parâmetro $K = 2$ com base em três pilares:

1. **Superioridade Métrica Absoluta:**
   * O ponto de pico do Coeficiente de Silhueta ocorre rigorosamente em $K = 2$ ($0{,}1427$). Observa-se uma degradação expressiva de separabilidade à medida que $K$ avança ($0{,}0880$ em $K=3$ e patamares próximos de zero ou negativos para $K \ge 4$).

2. **Comportamento da Inércia (WSSSE) e Dispersão Espacial:**
   * Embora o WSSSE apresente redução contínua ($9{,}86 \times 10^8 \to 9{,}02 \times 10^8$), a ausência de um "cotovelo" acentuado evidencia que o acréscimo de *clusters* subdivide o espaço vetorial contínuo sem criar fronteiras densas ou agrupamentos verdadeiramente autônomos. Esse comportamento é característico de bases de Big Data financeiras com forte expansão via *One-Hot Encoding* (127 posições esparsas) e variáveis numéricas sob cauda longa.

3. **Princípio da Parcimônia e Integridade Metodológica:**
   * A flutuação observada no WSSSE para $K=7$ ($9{,}28 \times 10^8$ vs $9{,}22 \times 10^8$ em $K=6$) indica instabilidade computacional na formação das partições em $K > 3$, possivelmente associada ao processo de inicialização, convergência ou à própria complexidade do espaço vetorial resultante. Pelo princípio da parcimônia, adota-se a solução bidimensional ($K=2$), postergando qualquer inferência sobre os atributos de negócio até a conclusão do *profiling* descritivo.

> **Decisão Final:** Adota-se **$K = 2$** para o treinamento definitivo do algoritmo K-Means.

In [10]:
# =====================================================================
# ETAPA 6: MODELAGEM COM K-MEANS
# Subetapas 6.3 & 6.4: Treinamento Definitivo (K=2) e Volumetria dos Clusters
# =====================================================================

from pyspark.ml.clustering import KMeans
from pyspark.sql import functions as F
import time

K_OPTIMO = 2

print(f"--- 6.3 & 6.4 Treinando o Modelo Definitivo K-Means com K = {K_OPTIMO} ---")
t0 = time.time()

# 1. Instanciação com convergência estrita
kmeans_final = KMeans(
    k=K_OPTIMO,
    featuresCol="features",
    predictionCol="cluster",
    seed=42,
    initMode="k-means||",
    maxIter=100,
    tol=1e-5
)

# 2. Ajuste do modelo sobre o df_prepared (que contém a coluna 'features')
modelo_final = kmeans_final.fit(df_prepared)
tempo_treino = time.time() - t0
print(f"Modelo final ajustado com sucesso em {tempo_treino:.2f} segundos!\n")

# 3. Aplicação da predição sobre o df_prepared
df_clusterizado = modelo_final.transform(df_prepared)

# 4. Cálculo da Distribuição Volumétrica dos Registros por Cluster
total_registros = df_clusterizado.count()

print("="*65)
print("--- DISTRIBUIÇÃO VOLUMÉTRICA DOS CLUSTERS (K = 2) ---")
print("="*65)

df_volumetria = df_clusterizado.groupBy("cluster") \
    .agg(
        F.count("*").alias("Qtd_Registros"),
        (F.count("*") / total_registros * 100).alias("Percentual_%")
    ) \
    .orderBy("cluster")

df_volumetria.select(
    "cluster",
    F.format_number("Qtd_Registros", 0).alias("Qtd_Registros"),
    F.format_number("Percentual_%", 2).alias("Percentual_%")
).show(truncate=False)

--- 6.3 & 6.4 Treinando o Modelo Definitivo K-Means com K = 2 ---
Modelo final ajustado com sucesso em 1036.78 segundos!

--- DISTRIBUIÇÃO VOLUMÉTRICA DOS CLUSTERS (K = 2) ---
+-------+-------------+------------+
|cluster|Qtd_Registros|Percentual_%|
+-------+-------------+------------+
|0      |1,633,158    |21.80       |
|1      |5,859,050    |78.20       |
+-------+-------------+------------+



### 6.4 Distribuição Volumétrica dos Clusters Ajustados

O modelo final K-Means com $K = 2$ particionou a totalidade da base (7.492.208 registros) em dois agrupamentos populacionais bem delimitados e estatisticamente representativos.

#### Tabela de Volumetria e Proporção da Base

| Cluster | Quantidade de Registros | Proporção Relativa (%) | Classificação Metodológica |
| :---: | :---: | :---: | :--- |
| **`Cluster 1`** | **5.859.050** | **78,20%** | **Grupo Majoritário** (Caracterização econômica pendente do *profiling*) |
| **`Cluster 0`** | **1.633.158** | **21,80%** | **Grupo Minoritário** (Caracterização econômica pendente do *profiling*) |
| **Total** | **7.492.208** | **100,00%** | **Integridade Total Preservada (0% descartado)** |

---

#### Diagnóstico Técnico da Partição Populacional

1. **Equilíbrio e Relevância Amostral:**
   * A partição não produziu grupos desidratados (ex.: $< 1\%$), o que confirma a solidez da solução binária. Ambos os grupos contêm massa crítica de dados ($\approx 5{,}86 \text{ milhões}$ e $\approx 1{,}63 \text{ milhão}$ de observações) para análises descritivas de alta precisão.
2. **Preservação de Integridade:**
   * 100% da base tratada e escalonada na Etapa 5 recebeu rotulagem do algoritmo no PySpark ML, sem ocorrência de falhas ou registros não atribuídos.

In [11]:
# =====================================================================
# ETAPA 6: MODELAGEM COM K-MEANS
# Subetapa 6.5: Profiling Descritivo dos Clusters (Médias de Negócio)
# =====================================================================

from pyspark.sql import functions as F

print("--- 6.5 Profiling Descritivo e Financeiro dos Clusters ---")

# Agregação das médias de risco, sigilo e variáveis monetárias desnormalizadas
df_profiling_financeiro = df_clusterizado.groupBy("cluster").agg(
    F.count("*").alias("Qtd_Registros"),

    # Ratios de Risco e Sigilo
    F.avg("pct_inadimplencia").alias("Media_Pct_Inadimplencia"),
    F.avg("pct_ativo_problematico").alias("Media_Pct_Ativo_Problematico"),
    F.avg("flg_sigilo_operacoes").alias("Percentual_Sigilo_Operacoes"),

    # Valores Monetários Médios Brutos (Em R$)
    F.avg("carteira_ativa").alias("Media_Carteira_Ativa"),
    F.avg("carteira_a_vencer").alias("Media_Carteira_A_Vencer"),
    F.avg("carteira_vencida").alias("Media_Carteira_Vencida"),
    F.avg("carteira_inadimplencia").alias("Media_Carteira_Inadimplencia"),
    F.avg("ativo_problematico").alias("Media_Ativo_Problematico")
).orderBy("cluster")

print("="*105)
print("--- TABELA DE PROFILING FINANCEIRO E DE RISCO POR CLUSTER ---")
print("="*105)

df_profiling_financeiro.select(
    "cluster",
    F.format_number("Qtd_Registros", 0).alias("Registros"),
    F.format_number(F.col("Percentual_Sigilo_Operacoes") * 100, 2).alias("Sigilo_%"),
    F.format_number(F.col("Media_Pct_Inadimplencia") * 100, 2).alias("Inadimplencia_%"),
    F.format_number(F.col("Media_Pct_Ativo_Problematico") * 100, 2).alias("Ativo_Problematico_%"),
    F.format_number("Media_Carteira_Ativa", 2).alias("Carteira_Ativa_R$"),
    F.format_number("Media_Carteira_Inadimplencia", 2).alias("Inadimplencia_R$"),
    F.format_number("Media_Ativo_Problematico", 2).alias("Ativo_Problematico_R$")
).show(truncate=False)

--- 6.5 Profiling Descritivo e Financeiro dos Clusters ---
--- TABELA DE PROFILING FINANCEIRO E DE RISCO POR CLUSTER ---
+-------+---------+--------+---------------+--------------------+-----------------+----------------+---------------------+
|cluster|Registros|Sigilo_%|Inadimplencia_%|Ativo_Problematico_%|Carteira_Ativa_R$|Inadimplencia_R$|Ativo_Problematico_R$|
+-------+---------+--------+---------------+--------------------+-----------------+----------------+---------------------+
|0      |1,633,158|25.21   |3.43           |9.14                |43,866,417.03    |551,742.83      |1,924,840.71         |
|1      |5,859,050|27.34   |18.61          |27.87               |14,926,661.39    |794,867.28      |1,409,395.64         |
+-------+---------+--------+---------------+--------------------+-----------------+----------------+---------------------+



### 6.5 Interpretação Econômica e Diagnóstico Financeiro dos Clusters

A extração das médias desnormalizadas das variáveis financeiras e dos indicadores de risco permitiu caracterizar, com base em evidências empíricas diretas, a estrutura latente capturada pelo algoritmo K-Means ($K = 2$).

#### Tabela de Profiling Financeiro e de Risco

| Indicador / Atributo de Negócio | Cluster 0 (Minorítario: 21,80%) | Cluster 1 (Majoritário: 78,20%) | Razão / Comparativo Relativo |
| :--- | :---: | :---: | :---: |
| **Volumetria de Registros** | 1.633.158 | 5.859.050 | $\approx 1 : 3,6$ |
| **Carteira Ativa Média (`carteira_ativa`)** | **R$ 43.866.417,03** | **R$ 14.926.661,39** | **$2,9\times$ maior no Cluster 0** |
| **Taxa Média de Inadimplência (`pct_inadimplencia`)** | **3,43%** | **18,61%** | **$5,4\times$ menor no Cluster 0** |
| **Taxa Média de Ativo Problemático (`pct_ativo_problematico`)** | **9,14%** | **27,87%** | **$3,0\times$ menor no Cluster 0** |
| **Carteira Inadimplente Média (`carteira_inadimplencia`)** | R$ 551.742,83 | R$ 794.867,28 | $1,4\times$ maior no Cluster 1 |
| **Ativo Problemático Médio (`ativo_problematico`)** | R$ 1.924.840,71 | R$ 1.409.395,64 | $1,4\times$ maior no Cluster 0 |
| **Incidência de Sigilo (`flg_sigilo_operacoes`)** | 25,21% | 27,34% | Neutra / Distribuição Transversal |

---

#### Achados Metodológicos e de Negócio

1. **Multidimensionalidade do Agrupamento:**
   * O algoritmo capturou conjuntamente a **exposição financeira** e a **qualidade de crédito**. O **Cluster 0** reúne operações de grande ticket médio ($\approx \text{R\$ } 43,87 \text{ mi}$) com baixíssima taxa de inadimplência relativa ($3,43\%$). Em contrapartida, o **Cluster 1** agrega operações pulverizadas ($\approx \text{R\$ } 14,93 \text{ mi}$) sob expressivo estresse de risco relativo ($18,61\%$ de inadimplência e $27,87\%$ de ativo problemático).

2. **Neutralidade da Restrição de Sigilo Estatístico:**
   * A restrição regulatória de sigilo ($\le 15$ operações por instituição) apresentou distribuição simétrica ($\approx 25,2\%$ vs $\approx 27,3\%$), confirmando que a trava estatística do BACEN afeta transversalmente a base de dados, não sendo o vetor discriminante primário da clusterização.

3. **Nomenclaturas Regulatórias Adotadas (Ad-Referendum do Profiling Categórico):**
   * **Cluster 0:** *Carteiras de Maior Exposição e Menor Risco Relativo*
   * **Cluster 1:** *Carteiras de Menor Exposição e Maior Risco Relativo*

O algoritmo K-Means identificou dois perfis predominantes no conjunto de dados do SCR.data. O primeiro corresponde a carteiras de maior exposição financeira associadas a menores níveis de risco relativo, enquanto o segundo agrupa carteiras de menor exposição média, porém com indicadores significativamente superiores de inadimplência e ativo problemático. A variável de sigilo estatístico apresentou comportamento semelhante entre os grupos, indicando que não foi determinante para a formação dos agrupamentos.

In [12]:
# =====================================================================
# ETAPA 6: MODELAGEM COM K-MEANS
# Subetapa 6.6: Profiling Categórico (Porte, Cliente e Modalidade)
# =====================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("--- 6.6 Profiling Categórico dos Clusters ---")

# Função auxiliar para cruzamento categórico e cálculo de percentuais intra-cluster
def gerar_profiling_categorico(df, col_categoria, top_n=10):
    window_spec = Window.partitionBy("cluster")

    df_res = df.groupBy("cluster", col_categoria) \
        .agg(F.count("*").alias("Qtd_Registros")) \
        .withColumn("Total_Cluster", F.sum("Qtd_Registros").over(window_spec)) \
        .withColumn("Percentual_%", (F.col("Qtd_Registros") / F.col("Total_Cluster")) * 100) \
        .orderBy("cluster", F.col("Qtd_Registros").desc())

    return df_res.limit(top_n)

# 1. Profiling por PORTE DO CLIENTE
print("\n" + "="*80)
print("--- DISTRIBUIÇÃO DE PORTE POR CLUSTER ---")
print("="*80)
df_porte = gerar_profiling_categorico(df_clusterizado, "porte", top_n=15)
df_porte.select(
    "cluster", "porte",
    F.format_number("Qtd_Registros", 0).alias("Qtd_Registros"),
    F.format_number("Percentual_%", 2).alias("Percentual_%")
).show(truncate=False)

# 2. Profiling por TIPO DE CLIENTE (PF / PJ)
print("\n" + "="*80)
print("--- DISTRIBUIÇÃO DE TIPO DE CLIENTE POR CLUSTER ---")
print("="*80)
df_cliente = gerar_profiling_categorico(df_clusterizado, "cliente", top_n=10)
df_cliente.select(
    "cluster", "cliente",
    F.format_number("Qtd_Registros", 0).alias("Qtd_Registros"),
    F.format_number("Percentual_%", 2).alias("Percentual_%")
).show(truncate=False)

# 3. Profiling por MODALIDADE OPERACIONAL
print("\n" + "="*80)
print("--- DISTRIBUIÇÃO DAS PRINCIPAIS MODALIDADES POR CLUSTER ---")
print("="*80)
df_modalidade = gerar_profiling_categorico(df_clusterizado, "modalidade", top_n=10)
df_modalidade.select(
    "cluster", "modalidade",
    F.format_number("Qtd_Registros", 0).alias("Qtd_Registros"),
    F.format_number("Percentual_%", 2).alias("Percentual_%")
).show(truncate=False)

--- 6.6 Profiling Categórico dos Clusters ---

--- DISTRIBUIÇÃO DE PORTE POR CLUSTER ---
+-------+----------------------------------+-------------+------------+
|cluster|porte                             |Qtd_Registros|Percentual_%|
+-------+----------------------------------+-------------+------------+
|0      |Acima de 20 salÃ¡rios mÃ­nimos    |182,070      |11.15       |
|0      |Mais de 5 a 10 salÃ¡rios mÃ­nimos |172,741      |10.58       |
|0      |Mais de 10 a 20 salÃ¡rios mÃ­nimos|164,335      |10.06       |
|0      |Mais de 3 a 5 salÃ¡rios mÃ­nimos  |158,040      |9.68        |
|0      |MÃ©dio                            |151,620      |9.28        |
|0      |Pequeno                           |151,520      |9.28        |
|0      |Mais de 1 a 2 salÃ¡rios mÃ­nimos  |141,306      |8.65        |
|0      |Mais de 2 a 3 salÃ¡rios mÃ­nimos  |139,026      |8.51        |
|0      |AtÃ© 1 salÃ¡rio mÃ­nimo           |112,013      |6.86        |
|0      |Micro                             |92,

### 6.7 Síntese e Nomenclatura Definitiva dos Clusters no SFN

#### Cluster 0 (21,80% da base | 1.633.158 registros)
* **Nomenclatura Oficial:** Carteiras de Crédito Estruturado, Longo Prazo e Baixo Risco Relativo (Crédito Rural / Imobiliário / High Value)
* **Atributos Monetários Médios:** Carteira Ativa de R$ 43,87 milhões.
* **Indicadores de Risco Relativo:** Inadimplência Média de 3,43% | Ativo Problemático Médio de 9,14%.
* **Drivers de Negócio:** Forte concentração em Crédito Rural (34,20%) e Financiamento Imobiliário (25,72%), tomados por PF de alta renda (>20 SM) e PJs de porte elevado. Garantias reais de alta liquidez mantêm o risco relativo em patamares mínimos.

#### Cluster 1 (78,20% da base | 5.859.050 registros)
* **Nomenclatura Oficial:** Carteiras de Crédito Pulverizado, Curto Prazo e Elevado Risco Relativo (Varejo / PMEs / Unsecured Credit)
* **Atributos Monetários Médios:** Carteira Ativa de R$ 14,93 milhões.
* **Indicadores de Risco Relativo:** Inadimplência Média de 18,61% | Ativo Problemático Médio de 27,87%.
* **Drivers de Negócio:** Operações de varejo pulverizado, capital de giro e crédito pessoal sem garantia real forte, abrangendo PMEs (12,55% Porte Pequeno) e tomadores de menor renda relativa.

# ETAPA 7: SÍNTESE EXECUTIVA, IMPLICAÇÕES REGULATÓRIAS E CONCLUSÃO FINAL

---

## 7.1 Objetivo e Contextualização do Estudo

O Sistema Financeiro Nacional (SFN) transaciona diariamente volumes massivos de operações de crédito cujo acompanhamento prudencial é consolidado pelo Banco Central do Brasil (BACEN) no **Sistema de Informações de Crédito (SCR)**.

Este trabalho aplicou técnicas de **aprendizado de máquina não supervisionado (algoritmo $K$-Means em arquitetura distribuída PySpark ML)** sobre **7.492.208 registros regulatórios** do SCR (período 2024–2025). O objetivo central consistiu em identificar partições estruturais latentes de risco institucional e segmentação de carteiras a partir dos dados do SFN, fornecendo uma taxonomia empírica defensável sob o ponto de vista estatístico e regulatório.

---

## 7.2 Síntese da Arquitetura do Pipeline e Desempenho Computacional
==================================================================================================
                             PIPELINE DE MACHINE LEARNING (PYSPARK)
==================================================================================================
 1. Ingestão & Audit    : 7.492.208 registros | 24 variáveis originais | 0% nulos / duplicatas
 2. Feature Engineering : log1p(monetárias) | Ratios de inadimplência e ativo problemático
                          Flag de Sigilo (flg_sigilo_operacoes <= 15)
 3. Espaço Vetorial     : 135 features esparsas (One-Hot Encoding + StandardScaler withMean=False)
 4. Otimização de K     : Experimentos K ∈ [2, 8] | Pico em K = 2 (Silhouette Score: 0,1427)

 5. Treinamento Final   : k-means|| (seed=42, maxIter=100, tol=1e-5) | Cache em DISK_ONLY
==================================================================================================

---

## 7.3 Caracterização Empírica dos Clusters Formados ($K = 2$)

A modelagem binária segregou o espaço multidimensional do SCR em duas populações estatisticamente representativas e associadas a perfis distintos de exposição e risco:

| Dimensão / Indicador | Cluster 0 (21,80% \| 1.633.158 ops) | Cluster 1 (78,20% \| 5.859.050 ops) |
| :--- | :---: | :---: |
| **Denominação Definitiva** | **Carteiras de Maior Exposição e Menor Risco Relativo** | **Carteiras de Menor Exposição e Maior Risco Relativo** |
| **Carteira Ativa Média** | **R$ 43.866.417,03** | R$ 14.926.661,39 |
| **Inadimplência Relativa Média** | **3,43%** | **18,61%** |
| **Ativo Problemático Relativo Médio**| **9,14%** | **27,87%** |
| **Carteira Inadimplente Absoluta Média**| R$ 551.742,83 | R$ 794.867,28 |
| **Ativo Problemático Absoluto Médio** | R$ 1.924.840,71 | R$ 1.409.395,64 |
| **Incidência de Sigilo ($\le 15$ ops)**| 25,21% | 27,34% |
| **Perfil Qualitativo Observado** | Predomínio de operações de crédito estruturado e de longo prazo, especialmente financiamentos rurais (34,20%), imobiliários (25,72%) e modalidades de financiamento com garantias reais (23,97%), associadas a clientes de maior porte e faixas de renda elevadas. | Operações predominantemente pulverizadas, associadas a menores exposições médias e maiores níveis de risco relativo, com maior participação de clientes de menor porte e PMEs. |

---

## 7.4 Implicações para a Gestão de Risco Regulatória e Institucional

1. **Adequação de Provisionamento e Modelagem Prudencial:**
   * A dispersão entre a inadimplência relativa do Cluster 1 ($18,61\%$) e do Cluster 0 ($3,43\%$) aponta para a importância da calibração diferenciada na constituição de Provisão para Créditos de Liquidação Duvidosa (PCLD). As operações do Cluster 1 demandam parametrizações com maiores taxas estimadas de severidade (*Loss Given Default* - LGD).
2. **Mitigação de Risco por Modalidade e Mecanismos de Proteção:**
   * A predominância das modalidades rurais e imobiliárias no Cluster 0 está associada a menores índices de inadimplência relativa, sugerindo a relevância dos mecanismos de mitigação de risco e colateralização tradicionalmente presentes nessas modalidades para a estabilidade da carteira.
3. **Estratégias Diferenciadas de Cobrança e *Recovery*:**
   * Embora o Cluster 1 concentre a maior taxa relativa de estresse financeiro, o Cluster 0 carrega um ticket absoluto expressivo de ativo problemático médio por registro (R$ 1,92 mi vs R$ 1,41 mi). Isso sugere que as estratégias de recuperação de crédito devem combinar fluxos automatizados em massa para o Cluster 1 e reestruturações de dívida individualizadas para o Cluster 0.

---

## 7.5 Limitações Metodológicas do Estudo

* **Nível de Agregação da Base Publicada:** O SCR disponibilizado publicamente pelo BACEN consolida informações por instituição, UF, cliente e modalidade, impedindo análises individualizadas ao nível de contrato (*loan-level*).
* **Mecanismos de Mascaramento Estatístico (Sigilo):** A restrição regulatória que oculta o detalhamento de agregados com 15 ou menos operações atinge $\approx 26\%$ da base analisada, impondo limites ao detalhamento de micro-segmentos institucionais.
* **Recorte Temporal Estático (*Cross-Sectional*):** Os resultados refletem o retrato pontual do período analisado (2024–2025), não capturando a migração de risco ao longo de ciclos macroeconômicos completos.

---

## 7.6 Trabalhos Futuros

1. **Avaliabilidade de Outros Algoritmos:** Testar a convergência e separabilidade vetorial via *Bisecting K-Means*, *Gaussian Mixture Models* (GMM) ou algoritmos baseados em densidade em subamostras controladas.
2. **Análise Temporal de Transição de Estados:** Implementar modelos longitudinais para monitorar a taxa de migração de registros entre os Clusters 0 e 1 ao longo dos trimestres regulatórios.
3. **Integração com Modelagem Supervisionada:** Utilizar o rótulo do cluster (`cluster`) como covariável em modelos preditivos de default (e.g., XGBoost ou regressão logística) para aferir ganhos na capacidade discriminatória ($Gini$ / $KS$).

---

## 7.7 Conclusão Final

A aplicação do algoritmo **K-Means** sobre o conjunto de dados do SCR (2024–2025) permitiu identificar dois grandes perfis estruturais de crédito no Sistema Financeiro Nacional.

O **Cluster 0** caracteriza-se por operações de maior exposição financeira média (R$ 43,87 mi) e menor risco relativo (inadimplência de 3,43%), fortemente associadas a modalidades estruturadas de financiamento rural, imobiliário e de capital. Por sua vez, o **Cluster 1** reúne carteiras de menor exposição média (R$ 14,93 mi), porém predominantes em quantidade de registros (78,20% da base) e responsáveis pelos maiores índices relativos de inadimplência (18,61%) e ativo problemático (27,87%).

Estes resultados demonstram de forma empírica que técnicas de aprendizado não supervisionado em ambiente distribuído PySpark ML são capazes de extrair padrões relevantes e regulatoriamente aderentes de segmentação de risco em bases públicas de Big Data do setor bancário brasileiro, fornecendo insumos quantitativos para análises exploratórias, acompanhamento prudencial e governança de crédito.

In [13]:
# =====================================================================
# ETAPA 7: PERSISTÊNCIA DO MODELO TREINADO
# =====================================================================
import os
import shutil

path_modelo = "./modelo_kmeans_scr"

# Remove o diretório anterior para permitir sobrescrita limpa sem erros
if os.path.exists(path_modelo):
    shutil.rmtree(path_modelo)

# Salva o modelo treinado usando o método nativo do Spark ML (save)
modelo_final.write().overwrite().save(path_modelo)

print(f"✅ Modelo K-Means (K=2) salvo com sucesso em: {path_modelo}")

✅ Modelo K-Means (K=2) salvo com sucesso em: ./modelo_kmeans_scr
